# Spot-a-hit, fast path: predict with the saved model

Same decisions as `QueipoDeLlano_Ignacio.ipynb` without retraining anything. It needs `train.xlsx`,
`test.xlsx` and `spot_a_hit_model.joblib` (written by the full notebook) in the same folder. The
features are rebuilt with exactly the same code, because the label-based ones are fitted on the
train labels; the 9 tree models, the 5 BiGRUs and the stack are loaded, not trained.

In [1]:
import os, re, warnings
QUICK = False
N_FOLDS = 5 if QUICK else 10
SEEDS = 1 if QUICK else 3        # seed bagging of the tree models
GRU_SEEDS = 1 if QUICK else 5
N_JOBS = max(1, (os.cpu_count() or 2) - 2)   # -1 makes LightGBM/CatBoost crawl on some machines

import numpy as np, pandas as pd
import lightgbm as lgb, catboost as cb, xgboost as xgb
import torch, torch.nn as nn
from scipy.sparse import hstack
from scipy.special import logit
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, log_loss, roc_auc_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import QuantileTransformer, StandardScaler
warnings.filterwarnings("ignore")

In [2]:
GAIN_HIT, LOSS_FLOP = 2.5 - 1.0, 1.0 - 0.1
BREAK_EVEN = LOSS_FLOP / (GAIN_HIT + LOSS_FLOP)          # 0.375

def profit(y, invest):
    invest = np.asarray(invest, bool)
    return GAIN_HIT * np.sum(invest & (y == 1)) - LOSS_FLOP * np.sum(invest & (y == 0))

def evaluate(name, y, p):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    r = dict(model=name, AUC=roc_auc_score(y, p), accuracy=accuracy_score(y, p > 0.5),
             logloss=log_loss(y, p), profit_M=profit(y, p > BREAK_EVEN))
    print(f"{name:<34} AUC {r['AUC']:.4f}  acc {r['accuracy']:.4f}  logloss {r['logloss']:.4f}  profit {r['profit_M']:.0f}M")
    return r

In [3]:
train, test = pd.read_excel("train.xlsx"), pd.read_excel("test.xlsx")
for df in (train, test):
    df["track"], df["artist"] = df["track"].astype(str), df["artist"].astype(str)
y = train["target"].values
print(train.shape, test.shape, "| missing values:", int(train.isna().sum().sum()), int(test.isna().sum().sum()),
      "| duplicated rows:", int(train.drop(columns="id").duplicated().sum()))
print("hit rate", y.mean().round(3), "(balanced classes)")
train.groupby("decade").target.agg(songs="size", hit_rate="mean").round(3).T

(10000, 20) (2000, 19) | missing values: 0 0 | duplicated rows: 0
hit rate 0.483 (balanced classes)


decade,1960,1970,1980,1990,2000,2010
songs,1426.000,1578.000,1651.000,1699.000,1756.000,1890.000
hit_rate,0.612,0.547,0.482,0.454,0.407,0.433


In [4]:
# 1) Test titles are masked. Letters that vanish from test titles (non-initial positions):
MASKED = set("abcehimosuv")
mask_text = lambda s: "".join("*" if ch in MASKED else ch for ch in s)
letters = list("abcdefghijklmnopqrstuvwxyz")
freq = lambda s: pd.Series(list("".join(w[1:] for t in s for w in t.split()))).value_counts(normalize=True)
survival = (freq(test.track).reindex(letters).fillna(0) / freq(train.track).reindex(letters)).sort_values()
print("letter survival test/train:", survival.round(2).to_dict())
star = lambda s: s.str.count(r"\*").sum() / s.str.len().sum()
print(f"titles with '*': train {train.track.str.contains('[*]').mean():.1%}, test {test.track.str.contains('[*]').mean():.1%}")
print(f"masked characters: test {star(test.track):.1%}, train with the same rule {star(train.track.map(mask_text)):.1%}")

# 2) The artist almost decides the label
art = train.groupby("artist").target.agg(["size", "mean"])
multi = art[art["size"] >= 2]
print(f"artists with 2+ songs: {len(multi)}, all-hit {int((multi['mean'] == 1).sum())}, all-flop "
      f"{int((multi['mean'] == 0).sum())} -> {multi['mean'].isin([0, 1]).mean():.1%} pure")
print(f"test songs whose exact artist is in train: {test.artist.isin(set(train.artist)).mean():.1%}")

# 3) The id only encodes the decade
print("id AUC overall", round(roc_auc_score(y, train.id), 3), "| within each decade",
      {d: round(roc_auc_score(g.target, g.id), 2) for d, g in train.groupby("decade")})

letter survival test/train: {'a': 0.0, 'b': 0.0, 'c': 0.0, 'e': 0.0, 'h': 0.0, 'o': 0.0, 'm': 0.0, 'i': 0.0, 'u': 0.0, 'v': 0.0, 's': 0.0, 'x': 0.86, 'y': 0.92, 'j': 0.93, 'k': 0.96, 'z': 0.96, 'l': 0.96, 'r': 0.97, 't': 0.98, 'd': 0.99, 'g': 1.0, 'w': 1.01, 'p': 1.04, 'n': 1.04, 'f': 1.05, 'q': 1.23}
titles with '*': train 0.0%, test 98.9%
masked characters: test 39.0%, train with the same rule 38.6%
artists with 2+ songs: 2015, all-hit 671, all-flop 1012 -> 83.5% pure
test songs whose exact artist is in train: 58.3%
id AUC overall 0.571 | within each decade {1960: 0.52, 1970: 0.52, 1980: 0.51, 1990: 0.48, 2000: 0.49, 2010: 0.51}


In [5]:
train["track_raw"] = train["track"]                      # only used to group covers in the CV
train["track"] = train["track"].map(mask_text)

TITLE_RE = re.compile(r"\s*[\(\[].*?[\)\]]|\s+-\s+.*$")
title_key = lambda t: TITLE_RE.sub("", t.lower()).strip()   # '(Remastered)', ' - Live' removed
GROUPS = train["track_raw"].map(title_key).values

def folds(y, k, seed=0, groups=None):
    # Stratified and grouped: versions of the same title never fall on both sides
    g = groups if groups is not None else np.arange(len(y))
    return list(StratifiedGroupKFold(k, shuffle=True, random_state=seed).split(np.zeros(len(y)), y, g))

In [6]:
AUDIO = ["danceability", "energy", "key", "loudness", "mode", "speechiness", "acousticness",
         "instrumentalness", "liveness", "valence", "tempo", "duration_ms", "time_signature",
         "chorus_hit", "sections"]
CONTINUOUS = ["danceability", "energy", "loudness", "speechiness", "acousticness", "instrumentalness",
              "liveness", "valence", "tempo", "duration_ms", "chorus_hit", "sections"]
masked_re = lambda words: re.compile("|".join(re.escape(mask_text(w)) for w in words), re.I)
FEAT_RE = re.compile(r"\b(?:feat\.?|featuring|ft\.?|with|x|vs\.?)\b|&|,|\band\b", re.I)
CLASSICAL_RE = re.compile(r"\b(?:op\.|bwv|k\.\s?\d|no\.\s?\d|sonata|concerto|symphon|allegro|adagio|andante|"
                          r"prelude|nocturne|etude|minor|major|act \w+|scene|aria|movement|mvt)\b", re.I)
ENSEMBLE_RE = re.compile(r"orchestra|quartet|trio|ensemble|choir|philharmonic|symphony|band|chorus|"
                         r"singers|brothers|sisters|boys|girls", re.I)
TR_FEAT_M = masked_re(["feat"])
CLASSICAL_M = masked_re(["op.", "bwv", "no. ", "sonata", "concerto", "symphon", "allegro", "adagio", "andante",
                         "prelude", "nocturne", "etude", "minor", "major", "aria", "movement"])
VERSION_M = masked_re(["remaster", "live", "version", "mix", "edit", "demo", "acoustic", "instrumental",
                       "mono", "stereo", "take"])
nonascii = lambda s: sum(ord(c) > 127 for c in s) / max(len(s), 1)

def unsupervised(train, test):
    al = pd.concat([train.drop(columns=["target", "track_raw"]), test], ignore_index=True)
    X = al[["decade"] + AUDIO].copy()
    dur = al.duration_ms / 1000
    X["log_dur"], X["chorus_frac"], X["sec_per_min"] = np.log(dur), al.chorus_hit / dur, al.sections / (dur / 60)
    X["dance_x_energy"] = al.danceability * al.energy
    X["acoustic_minus_energy"] = al.acousticness - al.energy
    X["vocal"] = (al.instrumentalness < 0.5).astype(int)
    for c in CONTINUOUS:                                   # a 1965 hit does not sound like a 2015 hit
        g = al.groupby("decade")[c]
        X[f"{c}_zdec"] = (al[c] - g.transform("mean")) / g.transform("std")
    art = al.groupby("artist")                             # how prolific, how long, how they sound
    X["art_n"], X["art_n_dec"] = art.artist.transform("size"), art.decade.transform("nunique")
    X["art_first_dec"] = art.decade.transform("min")
    X["art_span"] = art.decade.transform("max") - X["art_first_dec"]
    for c in ["danceability", "energy", "acousticness", "instrumentalness", "speechiness", "valence", "loudness"]:
        X[f"art_mean_{c}"] = art[c].transform("mean")
        X[f"dev_art_{c}"] = al[c] - X[f"art_mean_{c}"]
    X["art_std_energy"] = art.energy.transform("std").fillna(0)
    a, t = al.artist, al.track
    X["art_multi"], X["art_ensemble"] = a.str.contains(FEAT_RE).astype(int), a.str.contains(ENSEMBLE_RE).astype(int)
    X["art_the"] = a.str.lower().str.startswith("the ").astype(int)
    X["art_len"], X["art_words"], X["art_nonascii"] = a.str.len(), a.str.split().str.len(), a.map(nonascii)
    X["tr_len"], X["tr_words"], X["tr_nonascii"] = t.str.len(), t.str.split().str.len(), t.map(nonascii)
    X["tr_paren"] = t.str.contains(r"[\(\[]").astype(int)
    X["tr_dash"] = t.str.contains(r" - ").astype(int)
    X["tr_digits"] = t.str.count(r"\d")
    X["tr_feat"] = t.str.contains(TR_FEAT_M).astype(int)  # keyword flags in their masked spelling
    X["tr_classical"] = (t.str.contains(CLASSICAL_M) | a.str.contains(CLASSICAL_RE)).astype(int)
    X["tr_version"] = t.str.contains(VERSION_M).astype(int)
    X["tr_star_share"] = t.str.count(r"\*") / t.str.len().clip(lower=1)
    X["tr_upper"] = t.map(lambda s: sum(c.isupper() for c in s) / max(sum(c.isalpha() for c in s), 1))
    X["tr_colon"] = t.str.contains(":").astype(int)
    tl = t.str.lower().str.strip()
    X["tr_title_n"] = tl.map(tl.value_counts())            # same title by several artists: standards
    return X

U = unsupervised(train, test)
U_tr, U_te = U.iloc[:len(train)].reset_index(drop=True), U.iloc[len(train):].reset_index(drop=True)
print(U.shape[1], "unsupervised features")

72 unsupervised features


In [7]:
def inner(k, seed, n, y, groups):
    return StratifiedGroupKFold(k, shuffle=True, random_state=seed).split(np.zeros(n), y, groups)

def smoothed_rate(kf, yf, ka, prior, k):
    g = pd.DataFrame({"k": kf, "y": yf}).groupby("k").y.agg(["sum", "count"])
    s = pd.Series(ka)
    return s.map((g["sum"] + prior * k) / (g["count"] + k)).fillna(prior).values, s.map(g["count"]).fillna(0).values

def oof_apply(fn, n_fit, y_fit, seed, groups):
    # Generic out-of-fold wrapper: fn(fit_idx, apply_idx or None) -> predictions
    out = None
    for a, b in inner(5, seed, n_fit, y_fit, groups):
        r = fn(a, b)
        out = np.zeros((n_fit,) + np.shape(r)[1:]) if out is None else out
        out[b] = r
    return out, fn(np.arange(n_fit), None)

def target_encode(kf, yf, ka, k, seed, groups):
    kf, prior = np.asarray(kf), yf.mean()
    f = lambda a, b: np.column_stack(smoothed_rate(kf[a], yf[a], kf[b] if b is not None else ka, prior, k))
    fo, ao = oof_apply(f, len(kf), yf, seed, groups)
    return fo[:, 0], fo[:, 1], ao[:, 0], ao[:, 1]

SPLIT_RE = re.compile(r"\s+(?:featuring|feat\.?|ft\.?|with|x|vs\.?|and|duet with)\s+|\s*[,&/]\s*", re.I)
credited = lambda a: [p.strip().lower() for p in SPLIT_RE.split(a) if p and p.strip()] or [a.lower()]

def credited_encode(af, yf, aa, seed, groups, k=2):
    def enc(af, yf, aa):
        d = pd.DataFrame([(i, p) for i, a in enumerate(af) for p in credited(a)], columns=["i", "p"])
        d["y"] = yf[d.i]
        g = d.groupby("p").y.agg(["sum", "count"]); prior = yf.mean()
        rate = ((g["sum"] + prior * k) / (g["count"] + k)).to_dict()
        out = np.zeros((len(aa), 3))
        for j, a in enumerate(aa):
            r = [rate[p] for p in credited(a) if p in rate]
            out[j] = (max(r), np.mean(r), len(r)) if r else (prior, prior, 0)
        return out
    af = np.asarray(af)
    fo, ao = oof_apply(lambda a, b: enc(af[a], yf[a], af[b] if b is not None else np.asarray(aa)), len(af), yf, seed, groups)
    return fo.T, ao.T

KNN_COLS = ["danceability", "energy", "loudness", "speechiness", "acousticness", "instrumentalness",
            "liveness", "valence", "tempo", "duration_ms", "decade"]

def knn_feature(fit, yf, app, seed, groups):
    Xf, Xa = fit[KNN_COLS].values, app[KNN_COLS].values
    def f(a, b):
        m = make_pipeline(QuantileTransformer(n_quantiles=200), KNeighborsClassifier(50, weights="distance"))
        return m.fit(Xf[a], yf[a]).predict_proba(Xf[b] if b is not None else Xa)[:, 1]
    return oof_apply(f, len(Xf), yf, seed, groups)

def text_joint(tf, af, yf, ta, aa, seed, groups, C=2.0):
    # Title || artist in one char 2-5 + word 1-2 vocabulary
    txt_f = (tf.str.lower() + " || " + af.str.lower()).values
    txt_a = (ta.str.lower() + " || " + aa.str.lower()).values
    def f(a, b):
        v1 = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=2, sublinear_tf=True)
        v2 = TfidfVectorizer(analyzer="word", ngram_range=(1, 2), min_df=2, sublinear_tf=True)
        tb = txt_f[b] if b is not None else txt_a
        Xa = hstack([v1.fit_transform(txt_f[a]), v2.fit_transform(txt_f[a])]).tocsr()
        Xb = hstack([v1.transform(tb), v2.transform(tb)]).tocsr()
        return LogisticRegression(C=C, max_iter=3000).fit(Xa, yf[a]).predict_proba(Xb)[:, 1]
    return oof_apply(f, len(txt_f), yf, seed, groups)

def text_separate(tf, af, yf, ta, aa, seed, groups, C=3.0):
    # Masked titles and clean artist names are different alphabets: separate vocabularies
    tf, af, ta, aa = (s.str.lower().values for s in (tf, af, ta, aa))
    def f(a, b):
        vs = [(TfidfVectorizer(analyzer="char_wb", ngram_range=(1, 5), min_df=2, sublinear_tf=True), 0),
              (TfidfVectorizer(analyzer="char_wb", ngram_range=(1, 5), min_df=2, sublinear_tf=True), 1),
              (TfidfVectorizer(token_pattern=r"[^ ]+", ngram_range=(1, 3), min_df=1, sublinear_tf=True), 1),
              (TfidfVectorizer(token_pattern=r"[^ ]+", min_df=2, sublinear_tf=True), 0)]
        fit_txt = (tf[a], af[a])
        app_txt = (tf[b], af[b]) if b is not None else (ta, aa)
        Xa = hstack([v.fit_transform(fit_txt[s]) for v, s in vs]).tocsr()
        Xb = hstack([v.transform(app_txt[s]) for v, s in vs]).tocsr()
        return LogisticRegression(C=C, max_iter=4000).fit(Xa, yf[a]).predict_proba(Xb)[:, 1]
    return oof_apply(f, len(tf), yf, seed, groups)

def supervised(fit, yf, app, seed):
    G = fit.track_raw.map(title_key).values
    F, A = pd.DataFrame(index=range(len(fit))), pd.DataFrame(index=range(len(app)))
    def put(name, res):
        F[name], A[name] = res
    fa, aa = fit.artist.values, app.artist.values
    te_f, cnt_f, te_a, cnt_a = target_encode(fa, yf, aa, 5, seed, G)
    F["art_te"], F["art_seen"], A["art_te"], A["art_seen"] = te_f, cnt_f, te_a, cnt_a
    kf, ka = (fit.artist + "|" + fit.decade.astype(str)).values, (app.artist + "|" + app.decade.astype(str)).values
    F["artdec_te"], _, A["artdec_te"], _ = target_encode(kf, yf, ka, 5, seed, G)
    put("text_p", text_joint(fit.track, fit.artist, yf, app.track, app.artist, seed, G))
    F["art_te_k1"], _, A["art_te_k1"], _ = target_encode(fa, yf, aa, 1, seed, G)
    hf, _, ha, _ = target_encode(fa, yf, aa, 1e-9, seed, G)
    F["art_hits"], A["art_hits"] = hf * F.art_seen, ha * A.art_seen
    F["art_flops"], A["art_flops"] = F.art_seen - F.art_hits, A.art_seen - A.art_hits
    put("knn_p", knn_feature(fit, yf, app, seed, G))
    blank_f, blank_a = pd.Series("", index=fit.index), pd.Series("", index=app.index)
    put("artname_p", text_joint(blank_f, fit.artist, yf, blank_a, app.artist, seed, G))
    put("text5_p", text_separate(fit.track, fit.artist, yf, app.track, app.artist, seed, G))
    pf, pa = [credited(a)[0] for a in fa], [credited(a)[0] for a in aa]
    F["prim_te"], F["prim_seen"], A["prim_te"], A["prim_seen"] = target_encode(np.array(pf), yf, np.array(pa), 2, seed, G)
    (F["cred_max"], F["cred_mean"], F["cred_seen"]), (A["cred_max"], A["cred_mean"], A["cred_seen"]) = \
        credited_encode(fa, yf, aa, seed, G)
    return F, A

In [8]:
dev = "cuda" if torch.cuda.is_available() else "cpu"
join = lambda d: (d.track.str.slice(0, 60) + " || " + d.artist.str.slice(0, 40)).tolist()
s_tr, s_te = join(train), join(test)
idx = {c: i + 1 for i, c in enumerate(sorted(set("".join(s_tr + s_te))))}
L = 104
def enc(ss):
    X = np.zeros((len(ss), L), dtype=np.int64)
    for i, s in enumerate(ss):
        v = [idx[c] for c in s][:L]; X[i, :len(v)] = v
    return torch.tensor(X)
S_tr, S_te = enc(s_tr), enc(s_te)
D_tr = torch.tensor(((train.decade.values - 1960) // 10).astype(np.int64))
D_te = torch.tensor(((test.decade.values - 1960) // 10).astype(np.int64))

class CharGRU(nn.Module):
    def __init__(self, V):
        super().__init__()
        self.emb, self.dec = nn.Embedding(V + 1, 48, padding_idx=0), nn.Embedding(6, 8)
        self.gru = nn.GRU(48, 96, batch_first=True, bidirectional=True)
        self.drop = nn.Dropout(0.3)
        self.out = nn.Sequential(nn.Linear(4 * 96 + 8, 64), nn.ReLU(), nn.Dropout(0.3), nn.Linear(64, 1))
    def forward(self, x, d):
        m = (x > 0).unsqueeze(-1).float()
        h, _ = self.gru(self.drop(self.emb(x)))
        pooled = torch.cat([(h * m).sum(1) / m.sum(1).clamp(min=1), (h - 1e4 * (1 - m)).max(1).values], 1)
        return self.out(torch.cat([pooled, self.dec(d)], 1)).squeeze(-1)

def gru_fit_predict(Xa, Da, ya, Xb, Db, seed, epochs=14, bs=128, return_net=False):
    torch.manual_seed(seed)
    net = CharGRU(len(idx)).to(dev)
    opt = torch.optim.AdamW(net.parameters(), lr=3e-3, weight_decay=1e-4)
    n = len(ya)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=3e-3, total_steps=epochs * ((n + bs - 1) // bs))
    Xa, Da, ya = Xa.to(dev), Da.to(dev), torch.tensor(ya, dtype=torch.float32).to(dev)
    for _ in range(epochs):
        net.train(); perm = torch.randperm(n, device=dev)
        for i in range(0, n, bs):
            j = perm[i:i + bs]
            opt.zero_grad(); nn.functional.binary_cross_entropy_with_logits(net(Xa[j], Da[j]), ya[j]).backward()
            opt.step(); sched.step()
    net.eval()
    with torch.no_grad():
        p = torch.sigmoid(net(Xb.to(dev), Db.to(dev))).cpu().numpy()
    return (p, net) if return_net else p

In [9]:
import joblib
bundle = joblib.load("spot_a_hit_model.joblib")
assert bundle["gru_vocab"] == idx, "character vocabulary differs from the saved model"

F, A = supervised(train, y, test, seed=0)                  # same call as the full notebook
X_test = pd.concat([U_te, A], axis=1)[bundle["columns"]]
lg = lambda p: logit(np.clip(p, 1e-5, 1 - 1e-5))

TEST = {name: np.mean([m.predict_proba(X_test)[:, 1] for m in models], 0) for name, models in bundle["trees"].items()}
gru_p = []
for state in bundle["gru_states"]:
    net = CharGRU(len(idx)); net.load_state_dict(state); net.to(dev).eval()
    with torch.no_grad():
        gru_p.append(torch.sigmoid(net(S_te.to(dev), D_te.to(dev))).cpu().numpy())

treesT = np.mean([lg(TEST[m]) for m in TEST], 0)
gT, tT = lg(np.mean(gru_p, 0)), lg(X_test["text5_p"].values)
uT = (X_test["cred_seen"].values > 0).astype(float)
ST = np.column_stack([treesT, gT, tT, uT, treesT * uT, gT * uT, tT * uT])
p_test = bundle["stack"].predict_proba(ST)[:, 1]

decisions = pd.DataFrame({"id": test["id"], "p_hit": p_test.round(4), "invest": (p_test > BREAK_EVEN).astype(int)})
decisions.to_csv("decisions.csv", index=False)
same = ((p_test > BREAK_EVEN) == (bundle["p_test"] > BREAK_EVEN)).mean()
print(f"Invest in {decisions.invest.sum()} of {len(decisions)} -> decisions.csv | "
      f"same decisions as the full run: {same:.1%} | max |dp| {np.abs(p_test - bundle['p_test']).max():.1e}")

Invest in 1090 of 2000 -> decisions.csv | same decisions as the full run: 100.0% | max |dp| 6.1e-08
